In [1]:
import numpy as np
import random
from collections import Counter
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from scipy.spatial import KDTree
from tqdm import tqdm
from joblib import Parallel, delayed  # 用于并行处理

class KNNClassifier:
    def __init__(self, k=3, metric='euclidean'):
        self.k = k
        self.metric = metric

    def fit(self, X_train, y_train):
        self.X_train = X_train
        self.y_train = y_train

    def _compute_distances(self, x):
        if self.metric == 'euclidean':
            return np.sqrt(np.sum((self.X_train - x) ** 2, axis=1))  # 欧几里得距离
        elif self.metric == 'manhattan':
            return np.sum(np.abs(self.X_train - x), axis=1)  # 曼哈顿距离
        elif self.metric == 'chebyshev':
            return np.max(np.abs(self.X_train - x), axis=1)  # 切比雪夫距离
        else:
            raise ValueError(f"Unsupported metric: {self.metric}")

    def _predict(self, x):
        # 计算所有训练样本与测试样本的距离
        distances = self._compute_distances(x)
        # 获取距离最近的 k 个样本的索引
        k_indices = np.argsort(distances)[:self.k]
        # 获取 k 个最近邻样本的标签
        k_labels = self.y_train[k_indices]
        # 返回出现频率最高的标签
        return Counter(k_labels).most_common(1)[0][0]

    def predict(self, X_test):
        # 对每个测试样本预测
        return np.array([self._predict(x) for x in tqdm(X_test)])

# 加载 MNIST 数据集
mnist = fetch_openml('mnist_784', version=1)
X, y = mnist['data'], mnist['target'].astype(int)

# 确保数据是 numpy 数组
X = np.array(X)
y = np.array(y)

# 数据拆分
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 对 X_train 进行采样，仅保留部分训练数据
sample_fraction = 0.1  # 仅保留 10% 的训练数据
sample_size = int(len(X_train) * sample_fraction)
indices = np.random.choice(len(X_train), sample_size, replace=False)
X_train_sampled = X_train[indices]
y_train_sampled = y_train[indices]

# 定义 k 值和距离度量
k_values = [1, 3, 5, 10, 20]
metrics = ['euclidean', 'manhattan', 'chebyshev']

# 存储结果
results = {}

# 训练和评估 k-NN 分类器
for metric in metrics:
    results[metric] = []
    for k in k_values:
        knn = KNNClassifier(k=k, metric=metric)
        knn.fit(X_train_sampled, y_train_sampled)
        # for x in X_test:
        #     print(x)
        predictions = knn.predict(X_test)
        # 计算准确性
        accuracy = np.sum(predictions == y_test) / len(y_test)
        results[metric].append(accuracy)
        print(f'Metric: {metric}, k: {k}, Accuracy: {accuracy:.4f}')


100%|██████████| 14000/14000 [02:48<00:00, 82.91it/s]


Metric: euclidean, k: 1, Accuracy: 0.9356


100%|██████████| 14000/14000 [02:46<00:00, 84.04it/s]


Metric: euclidean, k: 3, Accuracy: 0.9360


100%|██████████| 14000/14000 [02:44<00:00, 85.29it/s]


...

Metric: chebyshev, k: 5, Accuracy: 0.6714


100%|██████████| 14000/14000 [02:28<00:00, 94.18it/s] 


Metric: chebyshev, k: 10, Accuracy: 0.6650


100%|██████████| 14000/14000 [02:28<00:00, 94.34it/s]

Metric: chebyshev, k: 20, Accuracy: 0.6454
